# B2-022-probabilistic-latent-models — Practice p06 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** multivariate-gaussian

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch

mu = torch.tensor([1.0, -2.0, 0.5], dtype=torch.float64)
sigma = torch.tensor([[2.0, 0.6, 0.2],
                      [0.6, 1.5, -0.3],
                      [0.2, -0.3, 1.0]], dtype=torch.float64)
x = torch.tensor([[1.0, -2.0, 0.5],
                  [2.0, -1.0, 0.0],
                  [0.0, 0.0, 0.0],
                  [-1.5, -3.0, 2.0]], dtype=torch.float64)
expected = torch.tensor([-3.1793203645, -3.7098186463, -5.4623959659, -6.3692902958], dtype=torch.float64)
sigma_bad = torch.tensor([[1.0, 2.0, 0.0], [2.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=torch.float64)
sigma_asym = torch.tensor([[2.0, 0.5, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=torch.float64)
ATOL = 1e-8
RTOL = 1e-8

## Solution

For each row, $\log N(x_i;\mu,\Sigma)=-\tfrac12\big(d\log2\pi+\log\det\Sigma+q_i\big)$ with $q_i=(x_i-\mu)^\top\Sigma^{-1}(x_i-\mu)$.

- $\log\det\Sigma$ comes from `torch.linalg.slogdet`; a sign other than $+1$ means $\Sigma$ is not positive definite (here `sigma_bad` has determinant $-3$), so it is rejected.
- All quadratic forms come from **one** batched solve: stack the centered rows as the columns of $D^\top$ ($d\times N$), solve $\Sigma Y=D^\top$, and take column-wise dot products $q_i=\sum_j D^\top_{ji}Y_{ji}$. No inverse is formed and there is no loop over rows.
- Row 0 equals $\mu$, so $q_0=0$ and its log-density is the pure normalizer.

In [ ]:
def gaussian_log_density(x, mu, sigma):
    if x.dim() != 2:
        raise ValueError("x must be 2-D (N, d)")
    d = x.shape[1]
    if mu.shape != (d,):
        raise ValueError("mu must have shape (d,)")
    if sigma.shape != (d, d):
        raise ValueError("sigma must have shape (d, d)")
    if not torch.allclose(sigma, sigma.T, atol=1e-12, rtol=0):
        raise ValueError("sigma must be symmetric")
    sign, logabsdet = torch.linalg.slogdet(sigma)
    if sign.item() != 1.0:
        raise ValueError("sigma must have positive determinant (slogdet sign +1)")
    centered_t = (x - mu).T                       # (d, N)
    solved = torch.linalg.solve(sigma, centered_t)  # one batched solve, (d, N)
    quad = (centered_t * solved).sum(dim=0)        # (N,)
    return -0.5 * (d * math.log(2 * math.pi) + logabsdet + quad)


out = gaussian_log_density(x, mu, sigma)
normalizer = -0.5 * (3 * math.log(2 * math.pi) + torch.linalg.slogdet(sigma)[1])
# Probe only (not part of the implementation): evaluate each row alone as a (1, 3) batch.
single_rows = torch.cat([gaussian_log_density(x[i:i + 1], mu, sigma) for i in range(x.shape[0])])


def raises_value_error(fn):
    try:
        fn()
    except ValueError:
        return True
    return False


bad_raises = raises_value_error(lambda: gaussian_log_density(x, mu, sigma_bad))
asym_raises = raises_value_error(lambda: gaussian_log_density(x, mu, sigma_asym))
print(out, normalizer.item(), bad_raises, asym_raises)

### Answer check

In [ ]:
assert out.shape == (4,) and out.dtype == torch.float64
assert torch.allclose(out, expected, atol=ATOL, rtol=RTOL)
assert torch.allclose(out[0], normalizer, atol=ATOL, rtol=RTOL)
assert torch.allclose(single_rows, out, atol=ATOL, rtol=RTOL)
assert bad_raises and asym_raises
assert torch.linalg.slogdet(sigma_bad)[0].item() == -1.0